# Session 2 · Day 2 - Assemble the Clean Feature Table

Day 1 covered each preparation technique on its own: loading the sources, auditing coordinates, resolving institutions without a shared id, aligning to the PSGC, and joining boundaries. This notebook chains them into **one clean, analysis-ready feature table** at the center grain, the single input that the center-recommendation, gap-analysis, and assessor notebooks all build on.

The data loads from the public TESDA bucket.

### What this notebook covers
1. Load the raw accreditation file and the geocoded institution master
2. Audit and repair coordinates, keeping a quality flag
3. Resolve each accreditation to an institution without a shared id
4. Align to the PSGC and clean the program, sector, and expiry fields
5. Aggregate from one row per accreditation to **one row per center**
6. Save the analysis-ready table for the Day 2 analyses

> **How to use this notebook**
> - ▶ **Run each cell in order** (`Shift+Enter`). Later cells depend on earlier ones.
> - ✏️ **Exercise cells** have `### START CODE HERE ###` … `### END CODE HERE ###`. Write
>   code between those lines only.
> - ✅ After an exercise, run the **check cell**. Fix the code until it prints a pass.
> - 🐞 **Spot-the-flaw cells** run a wrong approach on purpose, to observe and correct.
> - 🎛️ **Play-around cells** are open to tweak, change a value, re-run, see what happens.

### New to Python or pandas?

Anyone coming from another language can run `S2_commands_primer` first. It lays out the mental model in one page: a `DataFrame` is a table (a SQL table, an R `data.frame`, a spreadsheet), a `merge` is a `JOIN`, `NaN` is the missing value, and a chain of `.method().method()` reads left to right. The comments in the code cells point out each pandas idiom as it appears.

In [ ]:
# Run once per session (skip if already installed).
!pip -q install pandas pyarrow openpyxl geopandas shapely

In [ ]:
import os, tempfile, re
import numpy as np
import pandas as pd

BASE = "https://storage.googleapis.com/tesda-datasets/data/"
REF_DATE = pd.Timestamp("2026-07-15")   # the accreditation database's "as of" date
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 60)

WORK = tempfile.mkdtemp(prefix="s2d2_00_")   # the saved table lands here
print("working folder:", WORK)

## The raw inputs

Two files carry everything the feature table needs. The **accreditation file** lists one row for each center-qualification pair, with raw coordinates and no institution id. The **institution master** is the geocoded provider list from Day 1, carrying a `tesda_inst_id` and PSGC codes. A center with `city_municipality` missing came from the assessment source, which is the side to link the accreditation file against.

In [ ]:
AC_URL = BASE + "2026%20Accredited%20Compentency%20Assessment%20Center%20based%20on%20July%2015%2C%202026%20database.csv"
ac = pd.read_csv(AC_URL, dtype=str).apply(lambda s: s.str.strip())   # stray spaces, e.g. "TVET "
ac = ac.drop(columns=["Center Manager", "Phone"])                    # personal data, never kept

master = pd.read_parquet(BASE + "tesda_coordinates.parquet")
master["source_record"] = np.where(master["city_municipality"].isna(), "assessment", "training")   # np.where = vectorized if/else (SQL CASE)

print("accreditation rows:", ac.shape, "| institution master:", master.shape)
ac.head(3)

**Expected output:** `accreditation rows: (12871, 14) | institution master: (8007, 22)`, then the first three accreditation rows. One row is one accreditation, so a center appears on many rows.

## Coordinate audit and repair

The accreditation file carries raw latitude and longitude with reversible defects (swapped columns, a dropped decimal point) and unreversible ones (a placeholder such as `(0, 0)`, a pin outside the country). The audit from `S2_D1_03` assigns a `coord_quality` flag and keeps a coordinate only when it is valid or safely repairable. Everything downstream reads the flag and never trusts a raw coordinate on its own.

In [ ]:
LAT_R, LON_R = (4.2, 21.5), (116.0, 127.0)   # a generous Philippine bounding box

def to_num(s):
    try:
        return float(str(s).replace(",", "").strip())
    except (TypeError, ValueError):
        return np.nan

def in_range(v, r):
    return v == v and r[0] <= v <= r[1]   # v == v is False only for NaN, so this also screens out missing

def shift_decimal(s, r, min_digits):
    s = str(s).strip()
    if not re.fullmatch(r"-?\d+", s) or len(s.lstrip("-")) < min_digits:
        return None
    v = abs(int(s))
    hits = [v / 10**k for k in range(1, 12) if in_range(v / 10**k, r)]   # which power-of-10 shift lands inside the box
    return hits[0] if len(hits) == 1 else None   # keep the fix only when exactly one shift fits

def check_coords(lat_s, lon_s):
    la, lo = to_num(lat_s), to_num(lon_s)
    if la != la or lo != lo:                        return "missing", np.nan, np.nan   # la != la tests for NaN
    if in_range(la, LAT_R) and in_range(lo, LON_R): return "ok", la, lo
    if in_range(lo, LAT_R) and in_range(la, LON_R): return "swapped_fixed", lo, la
    fla = la if in_range(la, LAT_R) else shift_decimal(lat_s, LAT_R, 7)
    flo = lo if in_range(lo, LON_R) else shift_decimal(lon_s, LON_R, 8)
    ambiguous = (shift_decimal(lat_s, LON_R, 8) is not None
                 and shift_decimal(lon_s, LAT_R, 7) is not None)
    if fla is not None and flo is not None and not ambiguous:
        return "decimal_fixed", fla, flo
    if la == lo or (abs(la) < 4 and abs(lo) < 4):   return "placeholder", np.nan, np.nan
    return "out_of_ph", np.nan, np.nan

res = [check_coords(a, b) for a, b in zip(ac["Latitude"], ac["Longtitude"])]   # zip pairs the two columns row by row
ac["coord_quality"] = [r[0] for r in res]   # split the (flag, lat, lon) tuples into three columns
ac["lat"] = [r[1] for r in res]
ac["lon"] = [r[2] for r in res]
ac["coord_quality"].value_counts()

**Expected output:** about `ok 11328`, `swapped_fixed 1122`, `out_of_ph 268`, `decimal_fixed 77`, `missing 58`, `placeholder 18`. Over 1,100 rows had latitude and longitude swapped and 77 had a dropped decimal, now repaired. The placeholder and out-of-country rows keep no coordinate.

## Resolve centers without a shared id

The accreditation file has no institution id, but the master does. A normalized **name + address** key links them: lowercasing and stripping punctuation collapses spelling variants onto one key, and a merge against the master's assessment-source rows carries the `tesda_inst_id` across. The same `center_key` of upper-cased name and city marks the center grain the table will collapse to.

In [ ]:
def normalize(text):
    # lower-case and strip everything but letters and digits so punctuation and spacing stop blocking a match
    return text.fillna("").str.lower().str.replace(r"[^a-z0-9]", "", regex=True)

ac["center_key"] = (ac["Assessment Center"].str.upper().str.replace(r"\s+", " ", regex=True)   # .str runs a string op down the column
                    + " | " + ac["City Name"].str.upper())
ac["link_key"] = normalize(ac["Assessment Center"]) + "|" + normalize(ac["Address"])

assess = master[master["source_record"] == "assessment"].copy()
assess["link_key"] = normalize(assess["name"]) + "|" + normalize(assess["address"])
key_to_id = assess.drop_duplicates("link_key")[["link_key", "tesda_inst_id"]]   # one id per key so the join cannot multiply rows

ac = ac.merge(key_to_id, on="link_key", how="left", validate="many_to_one")   # LEFT JOIN; validate= raises if a key repeats on the right
print(f"distinct centers (name + city):           {ac['center_key'].nunique():,}")   # nunique = COUNT(DISTINCT ...)
print(f"accreditations linked to an institution:  {ac['tesda_inst_id'].notna().mean():.1%}")   # mean of a boolean column = the share that are True

**Expected output:** about `2,727` distinct centers and `89.2%` of accreditations linked to an institution. The unlinked rows keep no id and carry whatever the file itself records.

## PSGC region code

A linked accreditation can borrow the institution master's PSGC region code, the audited geographic label. Group the master by `tesda_inst_id` and map the code onto the accreditation rows.

In [ ]:
reg_code = (master.dropna(subset=["psgc_observed_region"])
            .groupby("tesda_inst_id")["psgc_observed_region"].first())   # one region code per institution id
ac["psgc_region"] = ac["tesda_inst_id"].map(reg_code)   # .map looks each id up in that Series, a keyed lookup like VLOOKUP
print(f"accreditations with a PSGC region code: {ac['psgc_region'].notna().mean():.1%}")

## Clean the program, sector, and expiry fields

Text fields carry casing and spacing noise, so one qualification can read three ways. Upper-casing the qualification and title-casing the sector, both with collapsed whitespace, settle them to one spelling. The expiry date parses to a real date. Every accreditation in this file is still valid as of the database date, so an active-versus-expired split adds nothing. The signal that matters for capacity planning is how soon each accreditation expires. Flag the ones expiring within 12 months.

In [ ]:
ac["qualification_clean"] = ac["Qualification"].str.upper().str.replace(r"\s+", " ", regex=True).str.strip()   # chained .str ops settle the spelling
ac["sector_clean"] = ac["Sector Title"].str.title().str.replace(r"\s+", " ", regex=True).str.strip()
ac["expires_on"] = pd.to_datetime(ac["Date of Expiry"], errors="coerce")   # text -> datetime; errors="coerce" turns junk into NaT
ac["expiring_soon"] = (ac["expires_on"] >= REF_DATE) & (ac["expires_on"] < REF_DATE + pd.DateOffset(months=12))   # & is elementwise AND; parenthesize each side

print(f"expiry dates parsed:  {ac['expires_on'].notna().mean():.1%}")
print(f"earliest expiry: {ac['expires_on'].min().date()} | latest: {ac['expires_on'].max().date()}")
print(f"accreditations expiring within 12 months: {int(ac['expiring_soon'].sum()):,}")

**Expected output:** expiry dates parse at `100.0%`, the earliest is `2026-07-22` and the latest `2028-10-27`, and about `5,913` accreditations expire within 12 months. The useful feature is the renewal horizon. No row is expired as of the database date.

## 🐞 Spot the flaw - counting rows as centers

The accreditation file is at the accreditation grain. Each center can appear on many rows. A count that forgets this treats every row as a separate center and inflates every center that assesses many qualifications. Run the naive count, see the gap, then collapse to one row per center.

In [ ]:
# naive: treat every accreditation row as a separate center.
naive_centers = len(ac)
# correct: a center is one (name, city); it can hold many accreditations.
true_centers = ac["center_key"].nunique()
print(f"rows in the accreditation file:  {naive_centers:,}")
print(f"distinct centers (name + city):  {true_centers:,}")
print(f"inflation if rows are counted as centers: {naive_centers - true_centers:,}")

The naive figure is more than four times the real one. The file holds 12,871 accreditations across about 2,727 centers, so any per-center feature has to group to `center_key` first. The next cell does exactly that.

## One row per center

Aggregation collapses the accreditation rows to the center grain. Identity and location take the first value (repaired within the audit, so a center's rows agree), counts use `nunique` for the distinct qualifications and sectors, and `size` keeps the raw accreditation count. A separate group over the expiring-soon rows gives the renewal-pressure feature.

In [ ]:
def first_valid(s):
    v = s.dropna()
    return v.iloc[0] if len(v) else np.nan   # first non-null in the group, or NaN when the group is all null

# groupby + agg = SQL GROUP BY; each line names an output column and how to reduce its rows
centers = (ac.groupby("center_key").agg(
        center_name=("Assessment Center", "first"),
        region=("Region Name", "first"),
        province=("Province Name", "first"),
        city=("City Name", "first"),
        psgc_region=("psgc_region", first_valid),
        lat=("lat", first_valid),
        lon=("lon", first_valid),
        coord_quality=("coord_quality", first_valid),
        tesda_inst_id=("tesda_inst_id", first_valid),
        n_qualifications=("qualification_clean", "nunique"),   # nunique = COUNT(DISTINCT qualification)
        n_sectors=("sector_clean", "nunique"),
        n_accreditations=("qualification_clean", "size"),      # size counts rows, duplicates included
    ).reset_index())   # turn the center_key group index back into a column

_soon = ac[ac["expiring_soon"]].groupby("center_key")["qualification_clean"].nunique()
centers["n_expiring_soon"] = centers["center_key"].map(_soon).fillna(0).astype(int)   # map the count back; centers with none become 0
centers["has_pin"] = centers["lat"].notna()
centers["linked"] = centers["tesda_inst_id"].notna()

print(f"feature table: {centers.shape[0]:,} centers x {centers.shape[1]} columns")
print(f"with a usable pin: {int(centers['has_pin'].sum()):,} | linked to master: {int(centers['linked'].sum()):,}")
centers.head(3)

**Expected output:** `feature table: 2,727 centers x 16 columns`, with about `2,646` carrying a usable pin and `2,220` linked to the master, then the first three center rows.

## ✏️ Exercise - pin coverage by region

A coverage map is only as honest as its coordinates. For **each region** (`region`), compute the share of centers that carry a usable pin (`has_pin` is true). Sort ascending so the worst-covered region comes first, and store the result in `pin_rate`.

In [ ]:
### START CODE HERE ###   (about 2 lines)
pin_rate = None    # per region: share of centers with a usable pin, sorted ascending
### END CODE HERE ###
print(pin_rate)

## ✅ Check the work

Run this. It recomputes the reference and compares it to `pin_rate`.

In [ ]:
try:
    ref = centers.groupby("region")["has_pin"].mean().sort_values()
    assert pin_rate is not None, "pin_rate is None - fill in the code."
    assert isinstance(pin_rate, pd.Series), f"expected a pandas Series, got {type(pin_rate)}"
    assert set(pin_rate.index) == set(ref.index), "some regions are missing."
    assert np.allclose(pin_rate.sort_index().values, ref.sort_index().values), "values do not match."
    assert list(pin_rate.index) == list(ref.index), "sort ascending (worst-covered region first)."
    print(f"✅ Passed! Lowest pin coverage: {ref.index[0]} at {ref.iloc[0]:.1%}.")
except AssertionError as e:
    print("❌ Not yet:", e)

### ❓ Question

After the assembly, `centers` holds one row per center. How many of those centers carry a usable pin (a coordinate the audit kept)?

In [ ]:
q_pinned = None   # <- answer here (an integer)

In [ ]:
_pinned = int(centers["has_pin"].sum())
if q_pinned == _pinned:
    print(f"✅ Correct! {_pinned:,} centers carry a usable pin.")
else:
    print(f"❌ Not yet: expected {_pinned}, got {q_pinned}.")

## Save the analysis-ready table

One clean table, two portable forms. The full table writes to **Parquet** with its types intact, and the located centers write to **GeoJSON** for mapping. These files are the single input that `S2_D2_01` (center recommendation), `S2_D2_02` (gap analysis), and `S2_D2_03` (assessor workload) read.

In [ ]:
import geopandas as gpd

FEATURES = ["center_key", "center_name", "region", "province", "city", "psgc_region",
            "lat", "lon", "coord_quality", "has_pin", "tesda_inst_id", "linked",
            "n_qualifications", "n_sectors", "n_accreditations", "n_expiring_soon"]
table = centers[FEATURES].copy()

pq_path = os.path.join(WORK, "center_feature_table.parquet")
table.to_parquet(pq_path, index=False)   # Parquet keeps each column's dtype, unlike CSV

pinned = table[table["has_pin"]]   # boolean-column mask: keep the rows where has_pin is True
gdf = gpd.GeoDataFrame(pinned, geometry=gpd.points_from_xy(pinned["lon"], pinned["lat"]), crs="EPSG:4326")   # lon (x) before lat (y)
gj_path = os.path.join(WORK, "center_feature_table.geojson")
gdf.to_file(gj_path, driver="GeoJSON")

print("Parquet:", pq_path, "->", table.shape)
print("GeoJSON:", gj_path, "->", gdf.shape[0], "located centers")
assert os.path.exists(pq_path) and os.path.exists(gj_path), "a file was not written"
print("Saved. This table is the input to S2_D2_01, S2_D2_02, and S2_D2_03.")

## 🎛️ Play around - the renewal horizon

The 12-month window is a choice. Move the horizon and watch how many centers hold an accreditation expiring within it, the pool a renewal drive would target.

In [ ]:
def centers_expiring_within(months):
    cut = REF_DATE + pd.DateOffset(months=months)
    soon = ac[(ac["expires_on"] >= REF_DATE) & (ac["expires_on"] < cut)]
    return soon["center_key"].nunique()

try:
    from ipywidgets import interact, IntSlider
    interact(lambda months=12: print(f"{centers_expiring_within(months):,} centers have an accreditation expiring within {months} months"),
             months=IntSlider(min=3, max=24, step=3, value=12))   # interact binds the slider to the lambda's months argument
except Exception:
    for m in (6, 12, 24):
        print(f"within {m} months: {centers_expiring_within(m):,} centers")

## Recap
- The accreditation file is at the accreditation grain. The feature table collapses it to one row per center.
- Coordinates are audited and repaired before use and carry a `coord_quality` flag.
- Institutions resolve through a normalized name + address key, since the files carry no shared id, so about nine in ten accreditations pick up a `tesda_inst_id` and a PSGC code.
- Program, sector, and expiry fields are cleaned. With no row expired, the useful date feature is the renewal horizon.
- The saved table is the single input to the Day 2 analyses.

**Next:** `S2_D2_01` ranks centers for a learner (center recommendation).

## References & further reading

- [pandas, group by](https://pandas.pydata.org/docs/user_guide/groupby.html)
- [pandas, merge and join](https://pandas.pydata.org/docs/user_guide/merging.html)
- [pandas, working with text data](https://pandas.pydata.org/docs/user_guide/text.html)
- [pandas, time series / dates](https://pandas.pydata.org/docs/user_guide/timeseries.html)
- [GeoPandas, writing files](https://geopandas.org/en/stable/docs/user_guide/io.html)
- [PSA, Philippine Standard Geographic Code (PSGC)](https://psa.gov.ph/classification/psgc)